# FinTok · Pipeline de texto modular

Este notebook prueba el pipeline final utilizando los módulos de `src/text/`.

Arquitectura:

`chunk_selector → fact_extractor → fact_checker → script_generator → JSON final`

La prueba usa un único ejemplo (`AAPL-2024-8-0043`) para comprobar el pipeline completo antes de automatizarlo sobre todo el corpus.


In [ ]:
!pip -q install -U "transformers>=4.51.0" accelerate bitsandbytes huggingface_hub pandas


## 1. Clonar FinTok y cargar el corpus

El notebook asume que los cuatro módulos `.py` están guardados en `src/text/` dentro del repositorio.


In [ ]:
from pathlib import Path
import sys
import json
import time
import zipfile
import pandas as pd
import torch

if not Path("/content/FinTok").exists():
    !git clone https://github.com/mirdbg/FinTok.git

REPO_DIR = Path("/content/FinTok")
ZIP_PATH = REPO_DIR / "data" / "corpus_miax_2026.zip"
WORK_DIR = Path("/content/corpus_miax_2026")

WORK_DIR.mkdir(parents=True, exist_ok=True)

assert ZIP_PATH.exists(), f"No encuentro: {ZIP_PATH}"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(WORK_DIR)

with open(WORK_DIR / "chunks.jsonl", "r", encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f if line.strip()]

df = pd.DataFrame(chunks)

# Permitir imports desde /content/FinTok
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f"Chunks cargados: {len(df):,}")
display(df[["chunk_id", "ticker", "fiscal_year", "item", "posicion"]].head())


## 2. Cargar Gemma 3 4B IT

El modelo se carga una sola vez en el notebook. Los módulos reciben después `ask_gemma` como función de generación, evitando que cada componente cargue su propia copia del modelo.


In [ ]:
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

MODEL_ID = "google/gemma-3-4b-it"

compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype="auto",
    quantization_config=quant_config,
)

model.eval()

print("Modelo cargado:", MODEL_ID)


In [ ]:
def ask_gemma(system_prompt, user_prompt, max_new_tokens=600):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    start = time.perf_counter()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    elapsed = time.perf_counter() - start
    generated = outputs[0][inputs["input_ids"].shape[-1]:]

    text = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    return text, elapsed


## 3. Importar los componentes del pipeline

Cada módulo tiene una responsabilidad única. El notebook solo orquesta las llamadas.


In [ ]:
from src.text.chunk_selector import get_context
from src.text.fact_extractor import extract_facts
from src.text.fact_checker import select_fact
from src.text.script_generator import generate_script, build_video_json


## 4. Elegir un ejemplo

Usamos como chunk de referencia `AAPL-2024-8-0043`. `get_context()` recupera automáticamente tres chunks consecutivos de la misma sección.


In [ ]:
CHUNK_ID = "AAPL-2024-8-0043"

context_df = get_context(
    df,
    chunk_id=CHUNK_ID
)

display(
    context_df[
        ["chunk_id", "ticker", "fiscal_year", "item", "posicion"]
    ]
)


## 5. Paso 1 · EXTRACT

Se extraen hasta tres ideas factuales, manteniendo evidencia y procedencia.


In [ ]:
extraction = extract_facts(
    context_df=context_df,
    generate_fn=ask_gemma
)

print(json.dumps(
    extraction["data"],
    indent=2,
    ensure_ascii=False
))


## 6. Paso 2 · SELECT / FACT CHECK

El editor decide si alguna idea es suficientemente relevante para FinTok. Si devuelve `DISCARD`, el pipeline debe detenerse y no generar un guion.


In [ ]:
selection = select_fact(
    context_df=context_df,
    extraction=extraction,
    generate_fn=ask_gemma
)

print(json.dumps(
    selection["data"],
    indent=2,
    ensure_ascii=False
))


## 7. Paso 3 · WRITE

Solo si la decisión es `KEEP`, Gemma genera el hook, la explicación y los dos highlights visuales.


In [ ]:
if selection["data"]["decision"].upper() == "KEEP":
    script = generate_script(
        context_df=context_df,
        selection=selection,
        generate_fn=ask_gemma
    )

    print(json.dumps(
        script["data"],
        indent=2,
        ensure_ascii=False
    ))
else:
    script = None
    print("DISCARD: no se genera script.")


## 8. Paso 4 · JSON final

Esta fase no utiliza IA. Python añade la intro y el outro fijos y construye las cuatro escenas con la estructura que consumirán después las fases de audio y vídeo.


In [ ]:
if script is not None:
    video_json = build_video_json(
        context_df=context_df,
        script=script
    )

    print(json.dumps(
        video_json,
        indent=2,
        ensure_ascii=False
    ))
else:
    video_json = None
    print("No hay vídeo porque el contexto fue descartado.")


## 9. Pipeline completo en una función

Una vez comprobadas las etapas individualmente, esta función permite ejecutar exactamente el mismo flujo con cualquier `chunk_id`.


In [ ]:
def run_example(chunk_id):
    context = get_context(df, chunk_id)

    extraction = extract_facts(
        context_df=context,
        generate_fn=ask_gemma
    )

    selection = select_fact(
        context_df=context,
        extraction=extraction,
        generate_fn=ask_gemma
    )

    if selection["data"]["decision"].upper() != "KEEP":
        return {
            "status": "DISCARD",
            "chunk_id": chunk_id,
            "selection": selection["data"]
        }

    script = generate_script(
        context_df=context,
        selection=selection,
        generate_fn=ask_gemma
    )

    return build_video_json(
        context_df=context,
        script=script
    )


### Ejecución end-to-end opcional

Esta celda vuelve a ejecutar las tres inferencias. No hace falta lanzarla si ya has probado las etapas 5–8; está preparada para comprobar el uso final de la API modular.


In [ ]:
# Descomenta cuando quieras probar la función end-to-end:
#
# result = run_example("AAPL-2024-8-0043")
# print(json.dumps(result, indent=2, ensure_ascii=False))
